This requests the homepage

In [ ]:
import requests

url = "https://quotes.toscrape.com/"
resp = requests.get(url, headers={"User-Agent": "SD321 student scraper"}, timeout=10, verify=False)

print(resp.status_code)        # Did the request succeed?
print(resp.text[:250])         # What do the first 250 characters look like?

In [ ]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib import robotparser   ##Is able to see the robot text if we are allowed to scrape the data
from urllib.parse import urljoin   ##is good for libraries

BASE = "https://quotes.toscrape.com/"
HEADERS = {"User-Agent": "SD321 student scraper"}

robots = robotparser.RobotFileParser(urljoin(BASE, "robots.txt"))
robots.read()

rows = []
url = BASE
while url and robots.can_fetch(HEADERS["User-Agent"], url):   # What quotes are on every allowed page?
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    for q in soup.select("div.quote"):
        rows.append({
            "text":   q.select_one("span.text").get_text(strip=True),
            "author": q.select_one("small.author").get_text(strip=True),
            "tags":   ";".join(a.get_text(strip=True) for a in q.select("a.tag")),
        })
    nxt = soup.select_one("li.next a")   ##Looks for the anchor point at the bottom to the next page
    url = urljoin(url, nxt["href"]) if nxt else None   ##this is if there are no more pages left
    time.sleep(1)                             # be polite: one request per second

raw = pd.DataFrame(rows)
raw.to_csv("quotes_raw.csv", index=False)     # save the raw scrape BEFORE changing anything
print(raw.shape)

Previous requests from the quotes page and pulls out the quotes
- time.sleep(1) prevents from overloading. does not push computer too hard


-Moving on to selenium and demo. Selenium renders Java so beautiful soup can see the tags on the page
- triggers all the js on the page. aka clicking all the buttons,
-sleniume outputs and cleans html data for beautiful soup

In [ ]:
js_url = "https://quotes.toscrape.com/js/"
js_resp = requests.get(js_url, headers=HEADERS, timeout=10)
js_soup = BeautifulSoup(js_resp.text, "html.parser")

print(len(js_soup.select("div.quote")))        # How many quote elements did requests receive?
print("var data" in js_resp.text)              # Is the quote data hiding in JavaScript code instead?


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()                    # opens a Chrome window (webdriver.Firefox() for Firefox)
driver.get(js_url)

# Wait up to 10 seconds for JavaScript to build at least one quote
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.quote")))

page = BeautifulSoup(driver.page_source, "html.parser")   # the HTML AFTER JavaScript ran
print(len(page.select("div.quote")))                       # How many quote elements now?
print(page.select_one("small.author").get_text())   

driver.find_element(By.CSS_SELECTOR, "li.next a").click()        # click the Next link
WebDriverWait(driver, 10).until(EC.url_contains("/page/2/"))       # wait for page 2 to load

page2 = BeautifulSoup(driver.page_source, "html.parser")
print(driver.current_url)                                          # Where are we now?
print(page2.select_one("small.author").get_text())                 # Who wrote the first quote on page 2?

driver.quit()                                                # Who wrote the first quote?

SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x61f0093bef7a <unknown>
#1 0x61f008bb2935 <unknown>
#2 0x61f008be1d1a <unknown>
#3 0x61f008bdfe39 <unknown>
#4 0x61f008c21613 <unknown>
#5 0x61f008beac27 <unknown>
#6 0x61f008be9f3e <unknown>
#7 0x61f008beaa11 <unknown>
#8 0x61f00938092c <unknown>
#9 0x61f00937f16b <unknown>
#10 0x61f009369e15 <unknown>
#11 0x61f00937fd9a <unknown>
#12 0x61f0093522d9 <unknown>
#13 0x61f0093a8648 <unknown>
#14 0x61f0093a87e5 <unknown>
#15 0x61f0093bd812 <unknown>
#16 0x73878149cb84 <unknown>
#17 0x738781529ecc <unknown>


Storing safely: 
need to normalize
load parents and keep keys then children